In [ ]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname

#RS
from torch.utils.data import WeightedRandomSampler



root_path = dirname(os.getcwd()) + "/SEPH_OUTCOME"

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print("CWD:", os.getcwd())
print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = "cpu"

CWD: /home/matteo/Documents/GNN-test2/SEPH_outcome
/home/matteo/Documents/GNN-test2/SEPH_outcome
/home/matteo/Documents/GNN-test2/SEPH_outcome/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_outcome/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_outcome/data/datasets/graphs_repair/


In [2]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [3]:
list(datasets_info.keys())

['BPI12_DECLINED_COMPLETE',
 'sepsis_cases_1',
 'sepsis_cases_4',
 'BPIC15_common']

In [ ]:
dataset = "BPI12_DECLINED_COMPLETE" #decide which dataset to work on

In [5]:
if dataset.startswith("BPIC15"):
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)["BPIC15_common"]
else:
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [6]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [7]:
tab_all = pd.read_csv(data_dir_processed+dataset+"_processed_all.csv")
tab_all.head()

,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,month,weekday,hour,open_cases,time:timestamp,Activity,case:AMOUNT_REQ,CaseID,Label
0,0.0,SCHEDULE,941.0,6.113517,24061.486817,46.0,11.0,4.0,15.0,758.0,1.321631e+09,wwijzigencontractgegevensschedule,25000.0,181447,False
1,0.0,SCHEDULE,942.0,0.741917,24062.228733,47.0,11.0,4.0,15.0,758.0,1.321631e+09,wwijzigencontractgegevensschedule,25000.0,181447,False
2,0.0,START,899.0,11766.656217,12505.183217,13.0,11.0,3.0,14.0,737.0,1.321542e+09,wnabellenoffertesstart,5000.0,183277,False
3,0.0,COMPLETE,435.0,975.723167,13480.906383,14.0,11.0,4.0,7.0,788.0,1.321601e+09,wnabellenoffertescomplete,5000.0,183277,False
4,0.0,START,570.0,15.111433,61129.463333,29.0,12.0,2.0,9.0,646.0,1.324460e+09,wvaliderenaanvraagstart,12850.0,183280,False


In [8]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [9]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
with open(data_dir_graphs + dataset + "_TEST_repair.pkl", "rb") as f:
    X_test = pickle.load(f)

In [10]:

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected, NormalizeFeatures

transform = ToUndirected()

with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for i in range(len(X_test)):
                X_test[i] = transform(X_test[i])
    


In [11]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_test)):
    n, edge_type = X_test[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)



In [12]:
node_types = list(node_types)
edge_types = list(edge_types)

In [13]:
node_types

['open_cases',
 'time:timestamp',
 'timesincecasestart',
 'case:AMOUNT_REQ',
 'lifecycle:transition',
 'event_nr',
 'weekday',
 'hour',
 'Activity',
 'timesincelastevent',
 'timesincemidnight',
 'month']

In [14]:
edge_types

[('hour', 'related_to', 'hour'),
 ('open_cases', 'related_to', 'open_cases'),
 ('Activity', 'related_to', 'timesincecasestart'),
 ('Activity', 'related_to', 'time:timestamp'),
 ('timesincecasestart', 'rev_related_to', 'Activity'),
 ('timesincemidnight', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'hour'),
 ('Activity', 'related_to', 'event_nr'),
 ('Activity', 'related_to', 'weekday'),
 ('month', 'related_to', 'month'),
 ('Activity', 'related_to', 'timesincemidnight'),
 ('Activity', 'related_to', 'case:AMOUNT_REQ'),
 ('Activity', 'related_to', 'open_cases'),
 ('time:timestamp', 'rev_related_to', 'Activity'),
 ('case:AMOUNT_REQ', 'rev_related_to', 'Activity'),
 ('timesincecasestart', 'related_to', 'timesincecasestart'),
 ('weekday', 'related_to', 'weekday'),
 ('open_cases', 'rev_related_to', 'Activity'),
 ('event_nr', 'rev_related_to', 'Activity'),
 ('time:timestamp', 'related_to', 'time:timestamp'),
 ('hour', 'rev_related_to', 'Activity'),
 ('lifecycle:transition', 'relat

## Hyperopt

In [15]:
print(f"PyTorch: {torch.__version__}")
#print(f"TorchVision: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch: 2.6.0+cu124
CUDA Available: False


In [16]:
from ax.service.managed_loop import optimize

In [17]:
from torch_geometric.nn import (
    HeteroConv,
    global_mean_pool,
    GATv2Conv,
    SAGEConv,
    TransformerConv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear
  )
from typing_extensions import Self

In [18]:
from torch_geometric.nn import HeteroConv, global_mean_pool, SAGEConv
from torch.nn import Module, ModuleList, Sequential, Linear, Dropout, BatchNorm1d, ReLU
import torch.nn.functional as F

class HGNN(Module):
    def __init__(self, nodes_relations, parameters):
        super().__init__()
        hid           = parameters["hid"]
        layers        = parameters["layers"]
        aggregation   = parameters["aggregation"]
        dropout_p     = parameters.get("dropout", 0.1)

        # 1) stack of hetero‐message‐passing layers
        self.convs = ModuleList()
        self.bns   = ModuleList()
        self.dps   = ModuleList()
        for _ in range(layers):
            # hetero‐conv over each relation
            conv = HeteroConv(
                { rel: SAGEConv((-1, -1), aggr=aggregation, out_channels=hid, normalize=False)
                  for rel in nodes_relations },
                aggr=aggregation,
            )
            self.convs.append(conv)
            # batchnorm + dropout for the hidden dim
            self.bns.append(BatchNorm1d(hid))
            self.dps.append(Dropout(dropout_p))

        # 2) final graph‐classification head: MLP hid→hid→1
        self.classifier = Sequential(
            Linear(hid, hid),
            ReLU(),
            BatchNorm1d(hid),
            Dropout(dropout_p),
            Linear(hid, 1),
        )

    def forward(self, batch):
        x_dict    = batch.x_dict
        edge_dict = batch.edge_index_dict

        # --- message‑passing with BN/ReLU/Dropout after each conv ---
        for conv, bn, dp in zip(self.convs, self.bns, self.dps):
            x_dict = conv(x_dict, edge_dict)

            # normalize + activate + drop only on the “Activity” embeddings
            act = x_dict["Activity"]
            act = bn(act)
            act = F.relu(act)
            act = dp(act)
            x_dict["Activity"] = act

            # for all other node types, just ReLU
            for nt, x in x_dict.items():
                if nt != "Activity":
                    x_dict[nt] = F.relu(x)

        # --- graph‑level readout on “Activity” nodes ---
        h_act  = x_dict["Activity"]
        pooled = global_mean_pool(h_act, batch["Activity"].batch)

        # --- final MLP head → logits ---
        logits = self.classifier(pooled).view(-1)
        return logits



    

In [19]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score
import torch.nn as nn
import time

In [20]:
#Weighted Random Sampling
#Pull out all labels into a single 1D tensor of 0/1
y_train = torch.cat([g.y for g in X_train]).long()
#count examples per class
class_counts = torch.bincount(y_train)
#Inverse frequency
class_weights = 1.0 / class_counts.float()

print("class_counts:", class_counts.tolist())
print("class_weights:", class_weights.tolist())


# number of negatives & positives
n_neg, n_pos = class_counts.tolist()

# the weight for positive class = n_neg / n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float, device=device)
print("Using BCEWithLogitsLoss pos_weight =", pos_weight.item())

#Assign each sample the weight of it's class
sample_weights = class_weights[y_train]
#create a sampler that draws 'len(sample_weights)' samples per epoch
sampler = WeightedRandomSampler(
     weights=sample_weights,
     num_samples=len(sample_weights),
     replacement=True,
 )

class_counts: [2088, 410]
class_weights: [0.0004789271915797144, 0.002439024392515421]
Using BCEWithLogitsLoss pos_weight = 5.092682838439941


In [21]:
from collections import Counter

# Draw 10,000 “indices” from the sampler
sampled_indices = list(WeightedRandomSampler(
    weights=sample_weights,
    num_samples=500,
    replacement=True
))

# Map each index back to its label
sampled_labels = [ y_train[idx].item() for idx in sampled_indices ]
print(Counter(sampled_labels))

Counter({0: 257, 1: 243})


In [ ]:
from copy import deepcopy
from tqdm.notebook import tqdm

def train_hgnn(config, epochs=20):
    
    print(config)

    net = HGNN(
        parameters=config,
        nodes_relations=edge_types,
    )
    net = net.to(device)

    # loss for graph binary classification
    #loss_fn = nn.BCEWithLogitsLoss()
    loss_fn = nn.BCEWithLogitsLoss()

    #train_loader = DataLoader(X_train, batch_size=config["batch_size"], shuffle=True)
    train_loader = DataLoader(
        X_train,
        batch_size=config["batch_size"],
        sampler=sampler,     # ← use the balanced sampler
        shuffle=False,       # ← don’t shuffle when using sampler
    )


    valid_loader = DataLoader(X_valid, batch_size=config["batch_size"], shuffle=False)


    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])

    best_model = None
    best_loss = float("inf")
    patience = 5
    pat_count = 0

    torch.cuda.empty_cache()

    for epoch in tqdm(range(0, epochs)):
        start_time = time.time()

        #print(f"Epoch: {epoch}\n")

        net.train()
        for _, x in enumerate(train_loader):
            x = x.to(device)

            optimizer.zero_grad()       

            logits = net(x) #shape [batch_size]
            labels = x.y.float() #shape [batch_size]
            loss = loss_fn(logits, labels)

            loss.backward()
            optimizer.step()

        #--validation--
        running_loss = 0.0
        correct = 0
        total = 0

        net.eval()
        with torch.no_grad():
            for x in valid_loader:
                x = x.to(device)
                logits = net(x)
                labels = x.y 

                running_loss +=loss_fn(logits, labels.float()).item()

                #compute binary predictions
                preds = (torch.sigmoid(logits) > 0.5).long()
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss = running_loss / len(valid_loader)
        val_acc = correct / total

        # Early stopping 
        if val_loss < best_loss:
            best_loss  = val_loss
            best_model = deepcopy(net)
            pat_count  = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                break

    return best_model

In [23]:
from torch_geometric.loader import DataLoader
import torch.nn as nn
import torch

def test_hgnn(net):
    """
    Evaluate a trained HGNN (graph‑level classifier) on X_test.
    Returns a dict with test loss and accuracy.
    """
    test_loader = DataLoader(X_test, batch_size=128, shuffle=False)
    loss_fn = nn.BCEWithLogitsLoss()

    net.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x in test_loader:
            x = x.to(device)
            logits = net(x)           
            labels = x.y              

            # accumulate loss
            total_loss += loss_fn(logits, labels.float()).item()

            # binary predictions & accuracy
            preds = (torch.sigmoid(logits) > 0.5).long()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(test_loader)
    accuracy = correct / total

    print(f"Test loss: {avg_loss:.4f}, Test accuracy: {accuracy:.4f}")
    return {"test_loss": avg_loss, "test_acc": accuracy}


In [24]:
# Calculate unique counts for categorical columns
list_unique = {col: len(tab_all[col].unique()) for col in categorical_columns}

#outputcat = {k : len(list_unique[k]) for k in list_unique}
outputcat = list_unique
outputreal = real_value_columns
print(outputcat)
print(outputreal)

{'Resource': 63, 'lifecycle:transition': 3, 'Activity': 36}
['timesincemidnight', 'timesincelastevent', 'timesincecasestart', 'event_nr', 'month', 'weekday', 'hour', 'open_cases', 'time:timestamp', 'case:AMOUNT_REQ']


In [25]:
def train_evaluate(config):
    trained_net = train_hgnn(config, epochs=50)
    return test_hgnn(trained_net)

In [26]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [27]:
y_train = torch.cat([batch.y for batch in X_train]).float()
num_true = y_train.sum().item()
num_false = len(y_train) - num_true

# Assign weights to BCEWithLogitsLoss
pos_weight = num_false / num_true
pos_weight = torch.tensor([num_false / num_true], device=device)

print("pos_weight: ", pos_weight)

pos_weight:  tensor([5.0927])


In [28]:
sample_config = {
    "hid":          128, #128
    "layers":       2, #2
    "lr":           1e-3,
    "batch_size":   256, #256
    "aggregation": "mean",
}


# 1-epoch train just to exercise the code-path and see prints
net = train_hgnn(sample_config, epochs=1)

# run the test (with your debug prints enabled)
res = test_hgnn(net)
print("test_hgnn returned:", res)


{'hid': 128, 'layers': 2, 'lr': 0.001, 'batch_size': 256, 'aggregation': 'mean'}


  0%|          | 0/1 [00:00<?, ?it/s]

Test loss: 15.5850, Test accuracy: 0.8131
test_hgnn returned: {'test_loss': 15.584960790780874, 'test_acc': 0.8130601792573624}


In [29]:
best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [64,128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "hid", "type": "choice", "values": [512], "value_type": "int", "is_ordered" : True,"sort_values":False},
        {"name": "layers", "type": "choice", "values": [2, 3, 4, 5], "value_type": "int", "is_ordered" : True, "sort_values":False},
        #{"name": "layers", "type": "choice", "values": [2], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-1], "value_type": "float", "log_scale": True},
        {"name": "batch_size", "type": "choice", "values": [128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        
        #{"name": "heads", "type": "choice", "values": [1,2], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "heads", "type": "choice", "values": [1], "value_type": "int", "is_ordered" : True,"sort_values":False},
        
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"}
        #{"name": "aggregation", "type" : "choice", "values" :["max"], "value_type" : "str"},
     
    ],
  
    evaluation_function=train_evaluate,
    objective_name='test_loss', #or can place 'test_acc'
    arms_per_trial=1,
    minimize = True,
    random_seed = 123,
    total_trials = 30
)

print(best_parameters)
means, covariances = values
print(means)
print(experiment)

/home/matteo/Documents/GNN-test2/SEPH_outcome/env_2/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False`  since the parameter is a string with more than 2 choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
/home/matteo/Documents/GNN-test2/SEPH_outcome/env_2/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return ChoiceParameter(
[INF

{'hid': 64, 'layers': 4, 'lr': 0.001671979996274695, 'batch_size': 512, 'aggregation': 'sum'}


  0%|          | 0/50 [00:00<?, ?it/s]

[INFO 06-18 18:30:32] ax.core.experiment: Attached data has some metrics ({'test_acc'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[INFO 06-18 18:30:32] ax.service.managed_loop: Running optimization trial 2...
[ERROR 06-18 18:30:32] ax.core.observation: Data contains metric test_acc that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric test_acc.
NoneType: None


Test loss: 0.5157, Test accuracy: 0.8131
{'hid': 256, 'layers': 2, 'lr': 0.027575328715146064, 'batch_size': 128, 'aggregation': 'max'}


...

  0%|          | 0/50 [00:00<?, ?it/s]

[INFO 06-18 19:51:55] ax.core.experiment: Attached data has some metrics ({'test_acc'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[ERROR 06-18 19:51:55] ax.core.observation: Data contains metric test_acc that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric test_acc.
NoneType: None
[ERROR 06-18 19:51:55] ax.core.observation: Data contains metric test_acc that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Igno

Test loss: 0.5102, Test accuracy: 0.8131


[WARNING 06-18 19:51:56] ax.modelbridge.cross_validation: Metric test_loss was unable to be reliably fit.
[WARNING 06-18 19:51:56] ax.service.utils.best_point: Model fit is poor; falling back on raw data for best point.
[WARNING 06-18 19:51:56] ax.service.utils.best_point: Model fit is poor and data on objective metric test_loss is noisy; interpret best points results carefully.


{'hid': 128, 'layers': 5, 'lr': 0.06674817360628986, 'batch_size': 128, 'aggregation': 'max'}
{'test_loss': 0.46548614593652576, 'test_acc': 0.8297055057618438}
Experiment(None)


In [30]:
from ax.service.utils.report_utils import exp_to_df

results = exp_to_df(experiment)

[WARNING 06-18 19:51:56] ax.service.utils.report_utils: Column reason missing for all trials. Not appending column.


In [31]:
results.sort_values(by="test_acc")

,trial_index,arm_name,trial_status,generation_method,test_acc,test_loss,hid,layers,lr,batch_size,aggregation
24,24,24_0,COMPLETED,BoTorch,0.186940,0.719145,64,3,0.000100,128,sum
17,17,17_0,COMPLETED,BoTorch,0.624200,0.650439,64,5,0.064933,128,max
12,12,12_0,COMPLETED,BoTorch,0.674136,0.676683,512,2,0.100000,128,mean
11,11,11_0,COMPLETED,BoTorch,0.797695,0.577291,128,4,0.100000,128,max
14,14,14_0,COMPLETED,BoTorch,0.809859,0.571407,512,5,0.100000,128,max
23,23,21_0,COMPLETED,BoTorch,0.809859,0.486640,64,5,0.100000,128,max
27,27,27_0,COMPLETED,BoTorch,0.813060,0.508221,256,5,0.100000,512,sum
26,26,26_0,COMPLETED,BoTorch,0.813060,0.482341,512,2,0.100000,512,sum
25,25,25_0,COMPLETED,BoTorch,0.813060,0.489242,512,2,0.000254,512,sum
22,22,22_0,COMPLETED,BoTorch,0.813060,0.491202,512,2,0.000100,128,sum


In [32]:
results = results.sort_values(by="test_acc")

In [33]:
results.to_csv(f"results/{dataset}.csv", sep=",")